# Week 4: Cross validation with "ccdefault.csv" Dataset

## Overview

This practice includes the following steps:

0. **Introduction**
1. **Data Loading and Preparation**
2. **Monte Carlo Cross Validation**
3. **10-Fold Stratified Cross-Validation**
4. **Conclusions**


# 0. Introduction


This practice applies the decision tree model to the `ccdefault.csv` dataset. The analysis compares repeated random train-test splits with 10-fold stratified cross-validation.

# 1. Data Loading and Preparation

In [7]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# read the dataset
path = '/content/drive/MyDrive/UIUC/IE 498 - Machine Learning in Finance/week6/ccdefault.csv'
data = pd.read_csv(path, index_col=False, na_values=['NA', 'Nan', 'N/A', 'na', 'n/a', 'null', 'NULL', 'None', 'none', ''])

print(f"{data.columns}")
data.head()

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'DEFAULT'],
      dtype='object')


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [8]:
# The target variable is DEFAULT
# Remove ID and separate the target variable

X = data.drop(columns=["ID", "DEFAULT"])
y = data["DEFAULT"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 23)
y shape: (30000,)


## 1.1. Initial 90/10 Train-Test Split

In [10]:
# Split 90% training and 10% test
# Use stratified sampling so that the proportion of the target classes is approximately the same in training and test.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (27000, 23)
Test set size: (3000, 23)


# 2. Monte Carlo Cross Validation

## 2.1. 10 Times Random Train-Test Splits

In [12]:
# Part 1: Random Train-Test Splits

# Store the accuracy scores
in_sample_scores = []
out_sample_scores = []

# Perform ten different 90/10 splits on X_train
for random_state in range(100, 100+10):

    # use 90% training set (X_train), do another 90/10 split, but do this ten times with different random_states
    X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_train, y_train, test_size=0.10,
        random_state=random_state, stratify=y_train)

    # Create a new decision tree for each split
    model_i = DecisionTreeClassifier(random_state=42)

    # Train the model
    model_i.fit(X_train_i, y_train_i)

    # In-sample prediction
    y_train_pred = model_i.predict(X_train_i)

    # Out-of-sample prediction
    y_test_pred = model_i.predict(X_test_i)

    # Calculate accuracy
    in_sample_accuracy = accuracy_score(y_train_i, y_train_pred)
    out_sample_accuracy = accuracy_score(y_test_i, y_test_pred)

    # Store scores
    in_sample_scores.append(in_sample_accuracy)
    out_sample_scores.append(out_sample_accuracy)

    # Display individual scores
    print(f"Random State {random_state}: "
        f"In-Sample Accuracy = {in_sample_accuracy:.4f}, "
        f"Out-of-Sample Accuracy = {out_sample_accuracy:.4f}")

Random State 100: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7222
Random State 101: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7393
Random State 102: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7278
Random State 103: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7189
Random State 104: In-Sample Accuracy = 0.9995, Out-of-Sample Accuracy = 0.7178
Random State 105: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7230
Random State 106: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7204
Random State 107: In-Sample Accuracy = 0.9995, Out-of-Sample Accuracy = 0.7311
Random State 108: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7048
Random State 109: In-Sample Accuracy = 0.9994, Out-of-Sample Accuracy = 0.7259


## 2.2. Calculate Mean and Standard Deviation

In [16]:
in_sample_mean = np.mean(in_sample_scores)
in_sample_std = np.std(in_sample_scores, ddof=1)

out_sample_mean = np.mean(out_sample_scores)
out_sample_std = np.std(out_sample_scores, ddof=1)

print("In-Sample Mean:", round(in_sample_mean, 4))
print("In-Sample Standard Deviation:", round(in_sample_std, 4))

print("\nOut-of-Sample Mean:", round(out_sample_mean, 4))
print("Out-of-Sample Standard Deviation:", round(out_sample_std, 4))

In-Sample Mean: 0.9994
In-Sample Standard Deviation: 0.0001

Out-of-Sample Mean: 0.7231
Out-of-Sample Standard Deviation: 0.0091


## 2.3. Final Evaluation on the Original Holdout Test Set

In [18]:
# Train the model on the entire 90% training set
final_model = DecisionTreeClassifier(random_state=42)
final_model.fit(X_train, y_train)

# Predict the original 10% holdout test set
y_test_pred = final_model.predict(X_test)

# Calculate final holdout accuracy
final_test_accuracy = accuracy_score(y_test, y_test_pred)

print("Final Holdout Test Accuracy:", round(final_test_accuracy, 4))

Final Holdout Test Accuracy: 0.7177


# 3. 10-Fold Stratified Cross-Validation

## 3.1. 10-Fold Cross-Validation Individual Fold Scores

In [21]:
# Part 2: Cross Validation

# Create the decision tree model
cv_model = DecisionTreeClassifier(random_state=42)

# Create 10-fold StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Run cross-validation on the original 90% training set
cv_scores = cross_val_score(cv_model, X_train, y_train, cv=skf, scoring="accuracy")

# Display Individual Fold Scores
print("Individual Fold Accuracy Scores:")
for fold, score in enumerate(cv_scores, start=1):
    print(f"Fold {fold}: {score:.4f}")

Individual Fold Accuracy Scores:
Fold 1: 0.7293
Fold 2: 0.7196
Fold 3: 0.7226
Fold 4: 0.7248
Fold 5: 0.7393
Fold 6: 0.7152
Fold 7: 0.7144
Fold 8: 0.7296
Fold 9: 0.7348
Fold 10: 0.7378


## 3.2. Calculate Mean and Standard Deviation

In [23]:
# Calculate Mean and Standard Deviation
cv_mean = np.mean(cv_scores)
cv_std = np.std(cv_scores, ddof=1)

print("Mean CV Accuracy:", round(cv_mean, 4))
print("Standard Deviation:", round(cv_std, 4))


Mean CV Accuracy: 0.7267
Standard Deviation: 0.0089


## 3.3. Evaluate Cross-Validation Model on Holdout Test Set

In [25]:
# Fit the model on the entire 90% training set
cv_final_model = DecisionTreeClassifier(random_state=42)
cv_final_model.fit(X_train, y_train)

# Predict the 10% test set
cv_test_pred = cv_final_model.predict(X_test)

# Calculate holdout accuracy
cv_test_accuracy = accuracy_score(y_test, cv_test_pred)

print("Holdout Test Accuracy:", round(cv_test_accuracy, 4))

Holdout Test Accuracy: 0.7177


#4. Conclusions

The two methods produced very similar estimates of the model's performance on unseen data. Monte Carlo Cross Validation had a mean out-of-sample accuracy of 0.7231, while 10-fold cross-validation had a slightly higher mean accuracy of 0.7267. The standard deviation was also slightly lower for 10-fold cross-validation (0.0089 versus 0.0091). The final holdout accuracy of 0.7177 was close to both estimates.

In conclusion, 10-fold cross-validation provides a slightly more stable estimate of test performance in this experiment.
In terms of computational efficiency, the difference is not pronounced in this experiment because both methods were run only 10 times. However, 10-fold cross-validation is more convenient to implement because the data are automatically divided into multiple folds, whereas Monte Carlo Cross Validation requires explicitly repeating the train-test split and model-fitting process in a loop.



In [ ]:
print("My name is Rong Gan")
print("My NetID is: ronggan2")
print("I hereby certify that I have read the University policy on Academic Integrity and that I am not in violation.")

My name is Rong Gan
My NetID is: ronggan2
I hereby certify that I have read the University policy on Academic Integrity and that I am not in violation.
